# 🏗️ LakeHouse Pipeline - Interactive Notebook

Este notebook permite ejecutar y explorar los Python scripts del proyecto de forma interactiva.

## Configuración del entorno

In [1]:
import pandas as pd


## 1. Ejecutar script de ingesta (Bronze Layer)

In [ ]:
df=pd

## 2. Explorar datos con DuckDB

Puedes consultar directamente los datos en el lakehouse.

In [ ]:
import duckdb

con = duckdb.connect()

# Configurar acceso a MinIO/S3
con.execute("""
    INSTALL httpfs; LOAD httpfs;
    SET s3_endpoint='minio:9000';
    SET s3_access_key_id='admin';
    SET s3_secret_access_key='password123';
    SET s3_region='us-east-1';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

print('✅ DuckDB conectado a MinIO')

In [ ]:
# Consultar la capa Bronze
df = con.execute("SELECT * FROM read_parquet('s3://lakehouse/bronze/sales_raw.parquet')").fetchdf()
df.head(10)

## 3. Visualización rápida

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ventas por cliente
df.groupby('customer')['amount'].sum().plot(kind='bar', ax=axes[0], color=['#6366f1', '#8b5cf6', '#a78bfa'])
axes[0].set_title('Ventas por Cliente')
axes[0].set_ylabel('Monto Total')
axes[0].tick_params(axis='x', rotation=0)

# Tendencia de ventas
df.sort_values('date').plot(x='date', y='amount', kind='line', ax=axes[1], marker='o', color='#6366f1')
axes[1].set_title('Tendencia de Ventas')
axes[1].set_ylabel('Monto')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Ejecutar otros scripts

Usa `%run` para ejecutar cualquier script del proyecto:

```python
# Ejemplos:
%run /workspace/python_scripts/ingest_bronze.py
%run /workspace/scripts/run_poc.sh
```

O usa `subprocess` para más control:

In [ ]:
import subprocess

# Ejemplo: ejecutar un script Python con captura de output
result = subprocess.run(
    ['python', '/workspace/python_scripts/ingest_bronze.py'],
    capture_output=True, text=True
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)
print('Return code:', result.returncode)

## 5. Ejecutar dbt desde el notebook

In [ ]:
# Ejecutar dbt run desde el notebook
result = subprocess.run(
    ['dbt', 'run', '--project-dir', '/workspace/dbt_project', '--profiles-dir', '/workspace/dbt_project'],
    capture_output=True, text=True,
    cwd='/workspace/dbt_project'
)
print(result.stdout)
if result.returncode != 0:
    print('⚠️ Errors:', result.stderr)